<a href="https://colab.research.google.com/github/pradeep200516/syntax_surgen/blob/main/The_Syntax_Surgeon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install black autopep8 transformers torch gradio reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 21.9 MB/s eta 0:00:00


In [ ]:
# ==============================
# Syntax Surgeon AI Code Fixer
# ==============================

import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import ast
import re
import black
import autopep8
from datetime import datetime
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

# ==============================
# Load Granite Model
# ==============================

MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# ==============================
# Python Code Detection
# ==============================

def detect_python_code(text):

    python_patterns = [
        r"def\s+\w+\(",
        r"class\s+\w+",
        r"import\s+\w+",
        r"for\s+\w+\s+in",
        r"print\(",
    ]

    for pattern in python_patterns:
        if re.search(pattern, text):
            return True

    return False


# ==============================
# Fix Python Syntax
# ==============================

def fix_python_code(code):

    try:
        ast.parse(code)
        valid = True
    except SyntaxError:
        valid = False

    try:
        code = autopep8.fix_code(code)
        code = black.format_str(code, mode=black.FileMode())
    except Exception:
        pass

    return code, valid


# ==============================
# AI Response
# ==============================

def ai_chat(user_message, history):

    if detect_python_code(user_message):

        fixed_code, valid = fix_python_code(user_message)

        response = "### Python Code Detected\n"

        if valid:
            response += "Code is syntactically correct.\n\n"
        else:
            response += "Syntax issues detected and fixed.\n\n"

        response += "### Fixed Code:\n"
        response += f"```python\n{fixed_code}\n```"

        return response

    else:

        messages = [
            {"role": "user", "content": user_message}
        ]

        result = generator(
            messages,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9
        )

        return result[0]["generated_text"]


# ==============================
# Export Chat to PDF
# ==============================

def export_chat(history):

    file_name = f"chat_export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"

    styles = getSampleStyleSheet()
    elements = []

    for user, bot in history:
        elements.append(Paragraph(f"<b>User:</b> {user}", styles["Normal"]))
        elements.append(Paragraph(f"<b>Bot:</b> {bot}", styles["Normal"]))
        elements.append(Spacer(1, 10))

    doc = SimpleDocTemplate(file_name)
    doc.build(elements)

    return file_name


# ==============================
# Gradio Interface
# ==============================

with gr.Blocks(title="Syntax Surgeon") as demo:

    gr.Markdown("# 🧠 Syntax Surgeon")
    gr.Markdown("AI Powered Code Fixer & Chatbot")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Enter your message or Python code")

    clear = gr.Button("Clear Chat")
    export = gr.Button("Export Chat to PDF")

    history = []

    def respond(message, chat_history):

        reply = ai_chat(message, chat_history)
        chat_history.append((message, reply))

        return "", chat_history


    msg.submit(respond, [msg, chatbot], [msg, chatbot])

    clear.click(lambda: None, None, chatbot, queue=False)

    export.click(export_chat, chatbot)

demo.launch()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

/tmp/ipykernel_352/1142643169.py:143: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_352/1142643169.py:143: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://167860926eab3eaef5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
